**Estudiante:** Jhonnatan Ezequiel Peñate Serrano  
**Curso:** KODIGO - Python Analyst  
**Tarea:** M4S16 - Laboratorio MLOps  
**Opcion elegida:** B (dataset sintetico propuesto)

---

# Laboratorio MLOps: Del Modelo Estatico al Servicio Productivo

En este laboratorio vas a construir una API para hacer predicciones en tiempo real, y vas a empaquetarla en un contenedor Docker para garantizar su portabilidad.


### ------ Opcion A - Genera o carga un dataset de tu elección y entrena el modelo que prefieras con él

In [1]:
# Carga y entrena tu modelo aqui

### ------ Opcion B - Utiliza el dataset sintético propuesto
**Propuesta de caso práctico:** una plataforma de streaming quiere anticipar que suscriptores estan por cancelar su plan (*churn*), usando tres datos de cada usuario:

- `meses_suscrito`: hace cuantos meses es suscriptor (entero)
- `gasto_mensual`: cuanto paga por mes, en dolares (decimal)
- `horas_uso_semanal`: cuantas horas por semana usa la plataforma (decimal)

El notebook sigue las mismas fases que se describen en la consigna del laboratorio. Segui el orden: cada fase depende de que la anterior este resuelta.

## Fase 1: Preparar el modelo

Antes de construir la API necesitamos algo que servir. Generamos un dataset sintetico y entrenamos un modelo simple de clasificacion. Esta parte ya viene resuelta porque el foco del laboratorio es la API y Docker, no el modelo en si.

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(7)
n = 500

meses_suscrito = np.random.randint(1, 48, n)
gasto_mensual = np.random.uniform(5, 60, n)
horas_uso_semanal = np.random.uniform(0, 30, n)

# Regla simulada: menos uso y menos antiguedad aumentan la probabilidad de cancelar
score = (horas_uso_semanal * -0.15) + (meses_suscrito * -0.05) + (gasto_mensual * 0.02) + np.random.normal(0, 1.0, n)
cancela = (score > score.mean()).astype(int)

df = pd.DataFrame({
    "meses_suscrito": meses_suscrito,
    "gasto_mensual": gasto_mensual,
    "horas_uso_semanal": horas_uso_semanal,
    "cancela": cancela
})

X = df[["meses_suscrito", "gasto_mensual", "horas_uso_semanal"]]
y = df["cancela"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)

print("Precision en test:", modelo.score(X_test, y_test))

Precision en test: 0.86


## Fase 2: Serializacion

**Tarea:** exporta el modelo entrenado a un archivo `model.pkl`, usando `joblib`. Despues, verifica que se puede volver a cargar sin errores.

In [3]:
import joblib

# Serializar el modelo entrenado en la Fase 1
joblib.dump(modelo, "model.pkl")
print("model.pkl guardado")

# Verificar que se puede volver a cargar sin errores
modelo_cargado = joblib.load("model.pkl")
print("Predicciones de prueba:", modelo_cargado.predict(X_test[:5]))
print("Precision del modelo cargado:", modelo_cargado.score(X_test, y_test))

model.pkl guardado
Predicciones de prueba: [1 0 0 1 1]
Precision del modelo cargado: 0.86


## Fase 3: Desarrollo de la API

**Tarea:** instala las librerias necesarias (`fastapi`, `uvicorn`, `pydantic`) y despues crea el archivo `main.py` con:

- Una clase `SuscriptorInput` (Pydantic) que defina los tres campos de entrada, con el tipo correcto de cada uno.
- Un endpoint `GET /health` que devuelva `{"status": "online"}`.
- Un endpoint `POST /predict` que reciba un `SuscriptorInput`, use el modelo cargado desde `model.pkl`, y devuelva la prediccion.

Fijate que el endpoint no solo debe decir si el suscriptor cancela o no: tiene que devolver un campo `riesgo` con el valor `"Alto"` si cancela, o `"Bajo"` si no.

In [4]:
!pip install -q fastapi uvicorn pydantic httpx

In [5]:
%%writefile main.py
"""API de prediccion de churn (cancelacion de suscriptores).

Sirve un modelo de regresion logistica serializado en model.pkl.
Endpoints:
    GET  /health   -> estado del servicio
    POST /predict  -> prediccion de riesgo de cancelacion
"""
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Ruta relativa al archivo, para que funcione igual en Colab y dentro de Docker
MODEL_PATH = Path(__file__).parent / "model.pkl"
modelo = joblib.load(MODEL_PATH)

app = FastAPI(title="API de Churn - Plataforma de Streaming")


class SuscriptorInput(BaseModel):
    """Datos de entrada de un suscriptor (validados por Pydantic)."""

    meses_suscrito: int = Field(..., ge=0, le=600, description="Meses como suscriptor")
    gasto_mensual: float = Field(..., ge=0, description="Gasto mensual en dolares")
    horas_uso_semanal: float = Field(..., ge=0, le=168, description="Horas de uso por semana")


@app.get("/health")
def health():
    """Verifica que el servicio esta en linea."""
    return {"status": "online"}


@app.post("/predict")
def predict(datos: SuscriptorInput):
    """Predice si el suscriptor cancelara su plan."""
    # DataFrame con los mismos nombres de columnas usados en el entrenamiento
    entrada = pd.DataFrame(
        [[datos.meses_suscrito, datos.gasto_mensual, datos.horas_uso_semanal]],
        columns=["meses_suscrito", "gasto_mensual", "horas_uso_semanal"],
    )
    cancela = int(modelo.predict(entrada)[0])
    return {
        "cancela": cancela,
        "riesgo": "Alto" if cancela == 1 else "Bajo",
    }

Writing main.py


**Tarea:** prueba tu API sin salir del notebook, usando `TestClient`. Proba el endpoint `/health` y el endpoint `/predict` con al menos un suscriptor de ejemplo.

In [6]:
from main import app
from fastapi.testclient import TestClient

client = TestClient(app)

# 1) Endpoint /health
r = client.get("/health")
print("/health ->", r.status_code, r.json())

# 2) Endpoint /predict con dos suscriptores de ejemplo
poco_uso = {"meses_suscrito": 2, "gasto_mensual": 55.0, "horas_uso_semanal": 1.0}
mucho_uso = {"meses_suscrito": 40, "gasto_mensual": 10.0, "horas_uso_semanal": 25.0}

for nombre, datos in [("poco uso", poco_uso), ("mucho uso", mucho_uso)]:
    r = client.post("/predict", json=datos)
    print(f"/predict ({nombre}) ->", r.status_code, r.json())

# 3) Validacion: un tipo de dato incorrecto debe rechazarse con 422
malo = {"meses_suscrito": "abc", "gasto_mensual": 10.0, "horas_uso_semanal": 5.0}
r = client.post("/predict", json=malo)
print("/predict (dato invalido) ->", r.status_code)

/health -> 200 {'status': 'online'}
/predict (poco uso) -> 200 {'cancela': 1, 'riesgo': 'Alto'}
/predict (mucho uso) -> 200 {'cancela': 0, 'riesgo': 'Bajo'}
/predict (dato invalido) -> 422


## Fase 4: Entorno y Dockerizacion

Ahora vamos a preparar los archivos para empaquetar la API en un contenedor. Recorda que Docker no corre dentro de Colab: aca preparamos los archivos, y el build/run se hace en tu propia computadora.

**Tarea:** crea el archivo `requirements.txt` con las librerias necesarias para que la API funcione dentro del contenedor.

In [7]:
# Genera requirements.txt con las versiones EXACTAS instaladas en este entorno
# (asi el modelo se carga igual dentro del contenedor)
from importlib.metadata import version

paquetes = ["fastapi", "uvicorn", "pydantic", "scikit-learn", "joblib", "pandas", "numpy"]
contenido = "\n".join(f"{p}=={version(p)}" for p in paquetes) + "\n"

with open("requirements.txt", "w") as f:
    f.write(contenido)

print(contenido)

fastapi==0.141.1
uvicorn==0.53.0
pydantic==2.13.5
scikit-learn==1.6.1
joblib==1.6.0
pandas==2.2.3
numpy==2.1.3



**Tarea:** escribe el `Dockerfile`, siguiendo la receta de la consigna:

1. Imagen base de Python (usa una version `-slim`).
2. Directorio de trabajo `/app`.
3. Copiar e instalar `requirements.txt`.
4. Copiar el resto del codigo (`main.py` y `model.pkl`).
5. Exponer el puerto 8000.
6. Comando de inicio con `uvicorn`, escuchando en `0.0.0.0`.

In [8]:
import sys

# Misma version de Python que este entorno, en variante -slim (imagen ligera)
py = f"{sys.version_info.major}.{sys.version_info.minor}"

dockerfile = f"""# Imagen base ligera de Python
FROM python:{py}-slim

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# Instalar dependencias primero (aprovecha la cache de capas de Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar solo el codigo y el modelo
COPY main.py model.pkl ./

# Puerto de la API
EXPOSE 8000

# Iniciar la API escuchando en todas las interfaces
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print(dockerfile)

# Imagen base ligera de Python
FROM python:3.13-slim

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# Instalar dependencias primero (aprovecha la cache de capas de Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar solo el codigo y el modelo
COPY main.py model.pkl ./

# Puerto de la API
EXPOSE 8000

# Iniciar la API escuchando en todas las interfaces
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]



## Fase 5: Pruebas locales

Con `main.py`, `model.pkl`, `requirements.txt` y `Dockerfile` listos, descarga los cuatro archivos a una carpeta en tu computadora y segui estos pasos en una terminal, con Docker instalado:

1. Construir la imagen:

```
docker build -t ml-api-app .
```

2. Correr el contenedor:

```
docker run -p 8000:8000 ml-api-app
```

3. Verificar el funcionamiento abriendo en el navegador:

```
http://localhost:8000/docs
```

Ahi deberias ver la documentacion interactiva de Swagger UI, con los endpoints `/health` y `/predict` listos para probar.

## Preparar la entrega

**Tarea:** organiza los cuatro archivos en una carpeta llamada `entrega_lab` y comprimela en un `.zip`, lista para subir a Moodle.

In [9]:
# LEEME.txt (opcional) con instrucciones basicas
leeme = """API de prediccion de churn - Laboratorio MLOps (KODIGO)
Autor: Jhonnatan Ezequiel Penate Serrano

Archivos:
- main.py          API FastAPI (GET /health, POST /predict)
- model.pkl        Modelo de regresion logistica serializado con joblib
- requirements.txt Dependencias con versiones fijas
- Dockerfile       Imagen Docker de la API

Como correr:
1. docker build -t ml-api-app .
2. docker run -p 8000:8000 ml-api-app
3. Abrir http://localhost:8000/docs

Ejemplo de entrada para /predict:
{"meses_suscrito": 2, "gasto_mensual": 55.0, "horas_uso_semanal": 1.0}
"""

with open("LEEME.txt", "w", encoding="utf-8") as f:
    f.write(leeme)
print("LEEME.txt creado")

LEEME.txt creado


In [10]:
import os
import shutil

os.makedirs("entrega_lab", exist_ok=True)

for archivo in ["main.py", "model.pkl", "requirements.txt", "Dockerfile", "LEEME.txt"]:
    shutil.copy(archivo, f"entrega_lab/{archivo}")

shutil.make_archive("entrega_lab", "zip", "entrega_lab")
print("Archivo generado: entrega_lab.zip")
print("Contenido:", os.listdir("entrega_lab"))

Archivo generado: entrega_lab.zip
Contenido: ['Dockerfile', 'requirements.txt', 'LEEME.txt', 'main.py', 'model.pkl']


In [11]:
# Si estas en Google Colab, esta celda descarga el zip a tu computadora
try:
    from google.colab import files
    files.download("entrega_lab.zip")
except ImportError:
    print("No estamos en Colab. Descarga entrega_lab.zip manualmente desde el panel de archivos.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Checklist antes de entregar

Antes de subir el `.zip` a Moodle, verifica cada punto:

- `main.py` define la clase `SuscriptorInput` con los tres campos correctos.
- `main.py` tiene los endpoints `GET /health` y `POST /predict`.
- `model.pkl` existe y corresponde al modelo entrenado (no esta vacio ni corrupto).
- `requirements.txt` incluye `fastapi`, `uvicorn` y `scikit-learn` como minimo.
- `Dockerfile` usa `0.0.0.0` (no `127.0.0.1`) y expone el puerto 8000.
- `docker build` termina sin errores en tu computadora.
- `docker run` deja la API corriendo, y `http://localhost:8000/docs` responde.
- (Opcional) agregaste un `LEEME.txt` con instrucciones basicas de como correr el proyecto.

Si todos los puntos estan marcados, tu entrega esta lista.